# Part 1: Train YOLO Object Detector

This notebook trains a YOLOv8 model to detect magical objects (book, glasses, wand, phone, pet).


In [ ]:
# Cell 1: Install dependencies
%pip install ultralytics roboflow


In [ ]:
# Cell 2: Download dataset
from roboflow import Roboflow

# Replace with your Roboflow API key
rf = Roboflow(api_key="YOUR_KEY")
project = rf.workspace("magical-objects").project("magical-objects")
dataset = project.version(1).download("yolov8")


In [ ]:
# Cell 3: Train model
from ultralytics import YOLO

# Load pretrained YOLOv8 model
model = YOLO('yolov8n.pt')  # 'n' for nano (fastest), 's' for small, 'm' for medium, etc.

# Train the model
results = model.train(
    data='magical_objects/data.yaml',  # Path to dataset config
    epochs=30,
    imgsz=640,
    batch=16,
    name='magical_objects_detector'
)


In [ ]:
# Cell 4: Test model
from PIL import Image

# Test on a sample image
img = 'test_image.jpg'  # Replace with your test image path
results = model(img)

# Show detections
results[0].plot()  # This will display the image with bounding boxes

# Print detected objects
for box in results[0].boxes:
    class_id = int(box.cls[0])
    confidence = float(box.conf[0])
    print(f"Detected: {results[0].names[class_id]} with confidence {confidence:.2f}")


In [ ]:
# Cell 5: Export and save model
# Export to TorchScript format (optional, for production)
model.export(format='torchscript')

# Copy the best model weights
import shutil
import os

# Find the best model from training
best_model_path = 'runs/detect/magical_objects_detector/weights/best.pt'

if os.path.exists(best_model_path):
    # Copy to project root or backend/data directory
    shutil.copy(best_model_path, '../backend/data/trained_model.pt')
    print(f"Model saved to: ../backend/data/trained_model.pt")
else:
    print("Best model not found. Check the runs/detect directory.")
